In [1]:
import pandas as pd
import numpy as np
import scipy
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn
print(np.__version__)
print(sklearn.__version__)
print(scipy.__version__)

1.26.4
1.2.2
1.10.1


In [2]:
df = pd.read_csv("cleaned_loan_data.csv")

df.head()

,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,0,5221545193,3915471226,7803938546,5,605,0
1,5,1958928726,822875252,2664843525,2,572,1
2,0,3363009259,202783085,6586671246,4,602,0
3,0,4766648001,2501730397,7435688347,5,612,0
4,1,1345827718,1768826187,2344832631,6,631,0


In [3]:
X = df.drop("default", axis=1)
y = df["default"]

In [19]:
print(df.columns)

Index(['credit_lines_outstanding', 'loan_amt_outstanding',
       'total_debt_outstanding', 'income', 'years_employed', 'fico_score',
       'default'],
      dtype='object')


In [20]:
print(y.value_counts())

default
0    8149
1    1851
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Loan_Model_Comparison")

<Experiment: artifact_location='mlflow-artifacts:/665849649108107952', creation_time=1775404835273, experiment_id='665849649108107952', last_update_time=1775404835273, lifecycle_stage='active', name='Loan_Model_Comparison', tags={}, workspace='default'>

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for c in [0.1, 1, 10]:

    with mlflow.start_run(run_name=f"logreg_C={c}"):

        model = LogisticRegression(C=c, max_iter=1000)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"C={c} → acc={acc:.3f}, recall={rec:.3f}, f1={f1:.3f}")

        # 🔥 paramètres
        mlflow.log_param("model", "logistic_regression")
        mlflow.log_param("C", c)

        # 🔥 métriques importantes
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)

        # 🔥 modèle
        mlflow.sklearn.log_model(model, "model")

2026/04/05 18:11:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


C=0.1 → acc=0.822, recall=0.003, f1=0.006


2026/04/05 18:11:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logreg_C=0.1 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/784f915c24494aed98ed11ae32d21794
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


2026/04/05 18:11:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


C=1 → acc=0.822, recall=0.003, f1=0.006


2026/04/05 18:11:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logreg_C=1 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/c9d1b5b6d5cb4a53883b3d094bb0ee9f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952
C=10 → acc=0.822, recall=0.003, f1=0.006


2026/04/05 18:11:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 18:11:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logreg_C=10 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/45b5188124a0456b9c637110dc23a859
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for depth in [3, 5, 10]:

    with mlflow.start_run(run_name=f"tree_depth={depth}"):

        model = DecisionTreeClassifier(max_depth=depth)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"depth={depth} → acc={acc:.3f}, recall={rec:.3f}, f1={f1:.3f}")

        # 🔥 paramètres
        mlflow.log_param("model", "decision_tree")
        mlflow.log_param("max_depth", depth)

        # 🔥 métriques importantes
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)

        # 🔥 modèle
        mlflow.sklearn.log_model(model, "model")

2026/04/05 18:13:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


depth=3 → acc=0.994, recall=0.977, f1=0.981


2026/04/05 18:13:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run tree_depth=3 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/3fbdabf6f9f84b369da783a568a9d688
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952
depth=5 → acc=0.995, recall=0.980, f1=0.984


2026/04/05 18:13:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 18:13:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run tree_depth=5 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/01b68dd46fb041fcbee747a4b0e48580
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


2026/04/05 18:13:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


depth=10 → acc=0.994, recall=0.980, f1=0.981


2026/04/05 18:13:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run tree_depth=10 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/d79f168084884192bedc0f19a5208969
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for n in [50, 100, 200]:

    with mlflow.start_run(run_name=f"rf_n={n}"):

        model = RandomForestClassifier(n_estimators=n)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"n={n} → acc={acc:.3f}, recall={rec:.3f}, f1={f1:.3f}")

        # 🔥 paramètres
        mlflow.log_param("model", "random_forest")
        mlflow.log_param("n_estimators", n)

        # 🔥 métriques importantes
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)

        # 🔥 modèle
        mlflow.sklearn.log_model(model, "model")

2026/04/06 20:21:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


n=50 → acc=0.995, recall=0.977, f1=0.986


2026/04/06 20:21:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run rf_n=50 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/5d8774be6e3b482cb8e3f2e08a76272e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


2026/04/06 20:21:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run rf_n=100 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/04928d276eb14bdca8d4e49312a4c491
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


2026/04/06 20:21:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


n=200 → acc=0.995, recall=0.977, f1=0.986


2026/04/06 20:21:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run rf_n=200 at: http://127.0.0.1:5000/#/experiments/665849649108107952/runs/601511b16c1c4bb8a534e6fb912c763d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/665849649108107952


In [9]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# meilleur modèle trouvé
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

# sauvegarde
joblib.dump(rf_model, "model.pkl")

['model.pkl']